# Assignment 3 – RAG Knowledge Assistant (Solution)


In [1]:
!pip install langchain langchain-google-genai langchain-community langchain-core langchain-text-splitters pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 10.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 63.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstal

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import getpass

In [2]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API Key")

Enter your API Key··········


In [3]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash',
    temperature=0 #deterministic answers -> less hallucination
)

## Part 1 — Data understanding & preprocessing

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("/content/troubleshooting_runbook.pdf")

In [6]:
documents=loader.load()

report=""
for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))
print("Total characters",len(report))

Total number of pages 7
Total characters 16583


In [7]:
print(report[:1000]) #first 1000 characters from the document

Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team  |  Version: 3.2  |  Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-service, search-service, reporting-engine and user-profile-api. It covers the ten most frequent
error codes seen in the incident log over the last two quarters. Each error code has a symptom description,
likely causes, a diagnostic checklist, a fix, and a prevention note.
2. How to use this runbook
Start from the error code in the alert or log line. Confirm the symptom matches, work through the
diagnostic checklist in order, and apply the fix only after the cause is confirmed. If two consecutive
diagnostic steps produce no signal,

### Text-quality checks

Page footers (Contoso Platform Engineering | ... | Page N) repeat on every page and would pollute chunks, so we strip them.

In [8]:
import re

for d in documents:
  txt=d.page_content
  print(f"page {d.metadata['page']+1}: {len(txt):5d} chars | {len(txt.split()):4d} words | footer present: {'Synthetic demo data' in txt}")

FOOTER=re.compile(r"Contoso Platform Engineering\s*\|[\s\S]*?Synthetic demo data\s*(Page\s*\d+)?")
for d in documents:
  d.page_content=FOOTER.sub(" ",d.page_content).strip()

print("\nFooter left after cleaning:",any('Synthetic demo data' in d.page_content for d in documents))

page 1:  2276 chars |  326 words | footer present: True
page 2:  1892 chars |  294 words | footer present: True
page 3:  2266 chars |  354 words | footer present: True
page 4:  2650 chars |  409 words | footer present: True
page 5:  2710 chars |  371 words | footer present: True
page 6:  2680 chars |  380 words | footer present: True
page 7:  2102 chars |  313 words | footer present: True

Footer left after cleaning: False


### Chunking

The runbook is short, so we use **chunk_size=1000** characters with **chunk_overlap=200**. This keeps one error-code section (symptom/causes/diagnostics/fix) mostly inside one chunk. Splitting prefers paragraph breaks, so section headings stay with their text.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #helps in chunking

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, #max characters in each chunk
    chunk_overlap=200 #characters overlapping between consecutive chunks
)

chunks=text_splitter.split_documents(documents)
print("Number of chunks:",len(chunks))
print("Avg chunk length:",sum(len(c.page_content) for c in chunks)//len(chunks))
print("\nSample chunk (page",chunks[10].metadata['page']+1,"):\n",chunks[10].page_content)

Number of chunks: 22
Avg chunk length: 832

Sample chunk (page 4 ):
 and communicate expected recovery time to stakeholders. Do not repeatedly restart healthy pods.
ERR-504 - Gateway timeout
Symptom: HTTP 504 after 30 seconds; p95 latency above 3,000 ms. The week of 10 August 2026 saw a
cluster of ERR-504 incidents on payment-gateway, traced to slow queries on the payments-db after an
index was dropped during a migration.
Likely causes: Slow database queries; thread pool saturation; a slow downstream call without a timeout;
an under-provisioned pod count after a traffic increase.
Diagnostics: 1. Check p95 latency per endpoint. 2. Run EXPLAIN ANALYZE on the slowest query (see
Section 5 database errors). 3. Check HTTP client timeouts: every outbound call must have an explicit
timeout of 5 seconds or less. 4. Check HPA status.
Fix: Restore any missing index, set explicit client timeouts, scale out replicas, and enable request hedging
for idempotent reads.


## Part 2 — Embeddings & vector database
- **Embedding model**: gemini-embedding-2 (Google Gemini) – converts every chunk into a dense vector so semantically similar text is close in vector space.
- **Vector store**: FAISS (in-memory, exact nearest-neighbour search) – ideal for a small corpus like this; each vector is stored with its chunk text and page metadata.

In [10]:
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

In [11]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print("Vectors stored:",vector_store.index.ntotal)

Vectors stored: 22


In [12]:
query="What causes ERR-401 and how do I fix it?"

results=vector_store.similarity_search_with_score(query=query,k=3)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"Score (L2 distance, lower = closer):{score}")
  print(f"Page:{doc.metadata['page']+1}")
  print(f"Document:{doc.page_content[:400]}\n")

RESULT0
Score (L2 distance, lower = closer):0.41504260897636414
Page:3
Document:Authentication and Rate Limiting Errors
ERR-401 - Expired or invalid authentication token
Symptom: Clients receive HTTP 401 with body {"error":"token_expired"}; spikes often follow a deployment
of auth-service or a change to token lifetime configuration. user-profile-api and payment-gateway are the
most affected consumers.
Likely causes: (a) access token TTL shortened in config; (b) clock skew abo

RESULT1
Score (L2 distance, lower = closer):0.5073598623275757
Page:3
Document:recent change to AUTH_TOKEN_TTL_SECONDS.
Fix: Restore the previous TTL (default 900 seconds), force a JWKS refresh by restarting the consumer
pods one at a time, and fix NTP drift if present. Ask client teams to enable proactive refresh 60 seconds
before expiry.
Prevention: Alert when 401 rate exceeds 2% for five minutes; require a canary check on every key
rotation.
ERR-429 - Rate limit exceeded


RESULT2
Score (L2 distance, lower = c

## Part 3 — Retrieval pipeline
Experiment with k = 3, 5, 7.

In [13]:
def show_retrieval(query,k):
  retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":k})
  docs=retriever.invoke(query)
  print(f"\n===== k={k} | query: {query}")
  for i,d in enumerate(docs):
    first_line=d.page_content.strip().split("\n")[0][:80]
    print(f"  {i+1}. page {d.metadata['page']+1} | {first_line}")

for q in ["Why is my service returning a 504 gateway timeout?",
          "A pod keeps restarting with exit code 137"]:
  for k in (3,5,7):
    show_retrieval(q,k)


===== k=3 | query: Why is my service returning a 504 gateway timeout?
  1. page 4 | and communicate expected recovery time to stakeholders. Do not repeatedly restar
  2. page 4 | Fix: Roll back the release. Then add a guard or default for the failing field, a
  3. page 4 | Application and Gateway Errors

===== k=5 | query: Why is my service returning a 504 gateway timeout?
  1. page 4 | and communicate expected recovery time to stakeholders. Do not repeatedly restar
  2. page 4 | Fix: Roll back the release. Then add a guard or default for the failing field, a
  3. page 4 | Application and Gateway Errors
  4. page 3 | recent change to AUTH_TOKEN_TTL_SECONDS.
  5. page 5 | (default 30 seconds). order-api and user-profile-api are the most frequent offen

===== k=7 | query: Why is my service returning a 504 gateway timeout?
  1. page 4 | and communicate expected recovery time to stakeholders. Do not repeatedly restar
  2. page 4 | Fix: Roll back the release. Then add a guard or default fo

**Observation:** k=3 normally already contains the section that answers a single-error-code question; k=5/7 add loosely related chunks (escalation, appendix) that cost tokens and can distract the LLM. Questions that span sections (e.g. *"Which team owns the service that has DB-POOL problems and how do I escalate?"*) benefit from k=5. We use **k=3** as default and **k=5** for multi-section questions.

In [14]:
retriever=vector_store.as_retriever(
    search_type="similarity", #similarity search between query embedding and chunk embeddings
    search_kwargs={"k":3} #top 3 chunks
)

## Part 4 — Answer generation (RAG)
Prompt design:
1. **Role** – on-call Senior Site Reliability Engineer.
2. **Grounding rule** – answer *only* from the retrieved context.
3. **Refusal rule** – if the answer is not in the context, say exactly *"I couldn't find this information"*.
4. **Output format** – 1. Answer and 2. Supporting Evidence (verbatim quotes + page numbers), so every claim can be checked against the source.
5. **temperature**=0 on the LLM to minimise hallucination.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

In [16]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Site Reliability Engineer and your job is to answer
    the user questions based ONLY on the retrieved context from the troubleshooting runbook.

    Do not use any outside knowledge.
    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context:{context}
    Question:{question}

    Provide:
    1. Answer (clear steps / bullet points; keep commands exactly as written in the context)
    2. Supporting Evidence (short verbatim quotes from the context with the page number)
    """
)

In [17]:
from langchain_core.output_parsers import StrOutputParser

In [18]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [19]:
def format_context(docs):
  return "\n\n".join([f"[Page {d.metadata['page']+1}]\n{d.page_content}" for d in docs])

def ask(question,k=3):
  retrieved_docs=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":k}).invoke(question)
  context=format_context(retrieved_docs)
  response=rag_chain.invoke({"context":context,"question":question})
  return response,retrieved_docs

### Sample queries and generated answers

In [20]:
question="What causes ERR-401 and how do I fix it?"
response,docs=ask(question)
print(response)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**1. Answer**

**Causes of ERR-401:**
* Access token TTL shortened in config.
* Clock skew above 30 seconds between services.
* Signing key rotated but the JWKS cache in the consumer was not refreshed.
* Refresh-token flow failing silently in the client SDK.

**How to Fix ERR-401:**
* Restore the previous TTL (default 900 seconds).
* Force a JWKS refresh by restarting the consumer pods one at a time.
* Fix NTP drift if present (diagnostics mention checking NTP sync using `chronyc tracking`).
* Ask client teams to enable proactive refresh 60 seconds before expiry.

---

**2. Supporting Evidence**

* **Page 3:** *"Likely causes: (a) access token TTL shortened in config; (b) clock skew above 30 seconds between services; (c) signing key rotated but the JWKS cache in the consumer was not refreshed; (d) refresh-token flow failing silently in the client SDK."*
* **Page 3:** *"Fix: Restore the previous TTL (default 900 seconds), force a JWKS refresh by restarting the consumer pods one at a tim

In [21]:
question="How do I find and clear the blocking session for a DB-1205 lock wait timeout?"
response,docs=ask(question)
print(response)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### 1. Answer
To find and clear the blocking session for a `DB-1205` lock wait timeout:

**Step 1: Find the blockers and blocked sessions**
Run the following PostgreSQL diagnostic query:

```sql
SELECT blocked.pid  AS blocked_pid, blocked.query AS blocked_query,
       blocking.pid AS blocking_pid, blocking.query AS blocking_query,
       now() - blocking.xact_start AS blocking_duration
FROM pg_stat_activity blocked
JOIN pg_stat_activity blocking
  ON blocking.pid = ANY(pg_blocking_pids(blocked.pid))
ORDER BY blocking_duration DESC;
```

**Step 2: Clear the blocking session**
Cancel the blocking session if it is an `idle-in-transaction` connection by running:

```sql
SELECT pg_cancel_backend(pid);
```
*(Replace `pid` with the `blocking_pid` identified from the diagnostic query).*

---

### 2. Supporting Evidence
* **[Page 5]** "Diagnostics:
 -- PostgreSQL: find blockers and blocked sessions
SELECT blocked.pid  AS blocked_pid, blocked.query AS blocked_query,
       blocking.pid AS block

In [22]:
question="What is the acknowledgement target and update cadence for a Sev1 incident, and who is it escalated to?"
response,docs=ask(question)
print(response)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Answer**

For a Sev1 incident:
* **Acknowledgement Target:** 5 min
* **Update Cadence:** Every 15 min
* **Escalate to:** Engineering Manager + Director on-call

---

**Supporting Evidence**

* **[Page 2]** "Sev1 | Full outage or data loss risk; revenue or security impact | 5 min | Every 15 min | Engineering Manager + Director on-call"


In [23]:
question="Which team owns order-api and what is its primary datastore and critical dependency?"
response,docs=ask(question,k=5)
print(response)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### 1. Answer
* **Owning team:** Team Atlas
* **Primary datastore:** PostgreSQL (orders-db)
* **Critical dependency:** inventory-service, payment-gateway

---

### 2. Supporting Evidence
* **Page 1:** 
  > "order-api Commerce Team Atlas PostgreSQL (orders-db) inventory-service, payment-gateway"


### Evidence of grounding
1. Every answer cites runbook text and page numbers.
2. Below we print the retrieved chunks used as context for one answer so the quotes can be matched.
3. An out-of-scope question must be refused, proving the model does not hallucinate from its own knowledge.

In [24]:
question="What is the MTTR target for Sev2 incidents?"
response,docs=ask(question)
print(response)
print("\n---- CONTEXT THE ANSWER WAS BASED ON ----")
for d in docs:
  print(f"[Page {d.metadata['page']+1}]",d.page_content[:300].replace("\n"," "),"...\n")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1. Answer
I couldn't find this information

2. Supporting Evidence
N/A

---- CONTEXT THE ANSWER WAS BASED ON ----
[Page 7] deploy-related incidents below 20 percent; zero repeat Sev1 incidents with the same root cause. D. Contacts Platform Reliability on-call: #sre-oncall. Security: #security-incidents. Database administration: #dba-help. Vendor escalation contacts are stored in the on-call wiki. This document is synthe ...

[Page 2] Severity and Escalation 5. Severity matrix Severity Definition Ack target Update cadence Escalate to Sev1 Full outage or data loss risk; revenue or security impact 5 min Every 15 min Engineering Manager + Director on-call Sev2 Major feature degraded for many users; workaround limited 10 min Every 30 ...

[Page 1] Identity Team Sentinel PostgreSQL (profiles-db) auth-service 4. Golden rules 1) Check recent deployments first: roughly one in four incidents in the log is deploy related, and a rollback is almost always faster than a forward fix. 2) Never restart

In [25]:
# Out-of-scope questions -> expected: "I couldn't find this information"
for q in ["What is the capital of France?",
          "How do I configure Kubernetes autoscaling for the billing-service?"]:
  response,_=ask(q)
  print("Q:",q,"\nA:",response,"\n"+"-"*60)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: What is the capital of France? 
A: 1. Answer:
I couldn't find this information

2. Supporting Evidence:
N/A (The provided context does not mention the capital of France.) 
------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: How do I configure Kubernetes autoscaling for the billing-service? 
A: **Answer:**
I couldn't find this information.

**Supporting Evidence:**
N/A (The provided context mentions `search-service`, `reporting-engine`, and `payment-gateway`, but does not contain any details or instructions regarding `billing-service` or how to configure Kubernetes autoscaling for it.) 
------------------------------------------------------------


## Part 5 — End-to-end RAG application

In [26]:
def rag_app(question,k=3,show_sources=True):
  response,docs=ask(question,k)
  print("QUESTION:",question,"\n")
  print(response)
  if show_sources:
    print("\nSOURCES:",sorted({d.metadata['page']+1 for d in docs}))
  print("="*80)

demo_questions=[
  "What should I check first when an alert fires?",
  "How do I fix DB-POOL connection pool exhaustion?",
  "How do I renew an expired TLS certificate?",
  "Why do pods get OOMKilled and how is it fixed?",
  "What should be in the post-incident checklist?",
]
for q in demo_questions:
  rag_app(q)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: What should I check first when an alert fires? 

**1. Answer**
* Check recent deployments first. 

*Note: Roughly one in four incidents is deploy-related, and performing a rollback is almost always faster than applying a forward fix.*

***

**2. Supporting Evidence**
* **[Page 1]**: "1) Check recent deployments first: roughly one in four incidents in the log is deploy related, and a rollback is almost always faster than a forward fix."

SOURCES: [1, 7]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: How do I fix DB-POOL connection pool exhaustion? 

### 1. Answer
To fix DB-POOL connection pool exhaustion, follow these steps:

* **Restart leaking pods:** Restart the leaking pods one at a time to release connections.
* **Patch the connection leak:** Update the code to patch the leak (use context managers).
* **Right-size the connection pool:** Ensure that `pool_size x replicas` is less than 80% of `max_connections`.
* **Use connection pooling middleware:** Consider PgBouncer in transaction mode for services with many replicas.

---

### 2. Supporting Evidence
* **[Page 5]**: "Fix: Restart the leaking pods one at a time to release connections, then patch the leak (use context managers). Right-size the pool: pool_size x replicas must be less than 80% of max_connections. Consider PgBouncer in transaction mode for services with many replicas."

SOURCES: [5]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: How do I renew an expired TLS certificate? 

**1. Answer**

To renew an expired TLS certificate, follow these steps:
* Renew the certificate through cert-manager: `kubectl cert-manager renew `
* Reload the ingress controller.
* Verify the full chain.
* If the ACME challenge fails, check DNS-01 credentials.

**2. Supporting Evidence**

* **[Page 6]**: "Fix: Renew the certificate through cert-manager (kubectl cert-manager renew ), reload the ingress controller, and verify the full chain. If the ACME challenge fails, check DNS-01 credentials."

SOURCES: [3, 6]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: Why do pods get OOMKilled and how is it fixed? 

### 1. Answer

**Why Pods Get OOMKilled (Likely Causes):**
* Memory limit is set below the real working set.
* Unbounded caches or result sets.
* Memory leak introduced by a release.
* JVM heap is configured larger than the container limit.
* A report is requested over an unusually large date range.

**How to Fix It:**
* Raise the memory limit by 25 to 50 percent as a stop-gap measure.
* Paginate or stream large results.
* Add max-range validation on report requests.
* Set JVM `-XX:MaxRAMPercentage=70`.

---

### 2. Supporting Evidence

* **[Page 6]** *"Likely causes: Memory limit set below real working set; unbounded caches or result sets; memory leak introduced by a release; JVM heap configured larger than the container limit; a report requested over an unusually large date range."*
* **[Page 6]** *"Fix: Raise the limit by 25 to 50 percent as a stop-gap, paginate or stream large results, add max-range validation on report req

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: What should be in the post-incident checklist? 

**1. Answer**

The post-incident checklist consists of the following:

* **Incident ticket configuration:** Ensure the ticket has a root cause category, error code, resolution time, and deploy-related flag.
* **Timeline reconstruction:** Reconstruct the timeline from alerts, deploys, and chat.
* **Blameless review:** Schedule a blameless review within five business days for every Sev1 and Sev2 incident.
* **Preventive action item:** Create at least one preventive action item with an assigned owner and due date.
* **Runbook update:** Update the runbook if a diagnostic step was missing or wrong.

---

**2. Supporting Evidence**

* **[Page 7]:** "B. Post-incident checklist 1. Incident ticket has root cause category, error code, resolution time and deploy-related flag. 2. Timeline reconstructed from alerts, deploys and chat. 3. A blameless review is scheduled within five business days for every Sev1 and Sev2. 4. At least one preven

In [27]:
# Interactive real-time Q&A (type 'exit' to stop)
while True:
  q=input("Ask the runbook assistant> ").strip()
  if q.lower() in ("exit","quit",""):
    break
  rag_app(q)

Ask the runbook assistant>  How does TLS certificate expired error will trigger?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: How does TLS certificate expired error will trigger? 

**Answer**

The "TLS certificate expired" error (`CERT-EXP`) triggers and manifests in the following way:

* **Client Connection Failure:** Clients fail when attempting to connect, resulting in `'certificate has expired'` or `'SSL handshake failed'` errors.
* **Service Impact:** It commonly hits `auth-service` because this service terminates TLS for many internal clients.
* **Immediate Failure (Low Latency):** Latency appears low because the SSL/TLS connections fail immediately upon connection attempt.
* **Underlying Causes/Triggers:** The condition is triggered when a certificate reaches its expiration (such as relying on a manual renewal calendar) or when automatic renewal fails (e.g., if the ACME challenge fails due to DNS-01 credential issues).

---

**Supporting Evidence**

* **[Page 6]:** *"CERT-EXP - TLS certificate expired"*
* **[Page 6]:** *"Symptom: Clients fail with 'certificate has expired' or 'SSL handshake f

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: what is the weather in Hyderabad right now? 

1. Answer
I couldn't find this information.

2. Supporting Evidence
N/A - The retrieved context contains SRE troubleshooting procedures for database/auth services, rate limits, and certificate management, but does not contain any information regarding the weather in Hyderabad.

SOURCES: [1, 3, 6]
Ask the runbook assistant> exit
